### Setup

In [ ]:
# Install dependencies
!pip install pyyaml pillow torchvision

import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
ZIP_PATH = "/content/drive/MyDrive/split_centroid_dataset/split_centroid_dataset.zip"
LOCAL_ROOT = "/content/dataset"
# ---------------------

# Extract Dataset
if os.path.exists(ZIP_PATH):
    print("Extracting Centroid Dataset...")
    if not os.path.exists(LOCAL_ROOT):
        os.makedirs(LOCAL_ROOT)
    !unzip -qo {ZIP_PATH} -d {LOCAL_ROOT}
    print("Extraction complete.")
else:
    print(f"Error: {ZIP_PATH} not found.")

### Architecture and Preprocessing

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import glob
import yaml

# --- BACKBONE: MobileNetV2 alpha=0.35 from a local ImageNet-pretrained checkpoint ---
# We use alpha=0.35 (not torchvision's default 1.0), initialised from an
# ImageNet-pretrained mobilenetv2_0.35-*.pth (from the repo's etc/ folder).
# Upload that file to your Drive and point MNV2_035_WEIGHTS_PATH at it. The file
# uses a non-torchvision MobileNetV2 layout (flat Conv/BN/ReLU6 modules), so its
# keys must be remapped to torchvision's fused ConvBNReLU layout before loading.
MNV2_035_WEIGHTS_PATH = "/content/drive/MyDrive/mobilenetv2_0.35-b2e15951.pth"


def _remap_mobilenet_v2_keys(state_dict):
    """Non-torchvision MobileNetV2 state_dict -> torchvision ConvBNReLU key layout.

    Only the feature extractor is kept; the classifier and the final 1x1 conv
    (features.18 / `conv.*`) are dropped because FOMO cuts the backbone before
    them. ReLU6 layers carry no parameters and are skipped.
    """
    remapped = {}
    for key, value in state_dict.items():
        if not key.startswith("features.") or key.startswith("features.18"):
            continue
        parts = key.split(".")
        block = parts[1]
        if block == "0" or parts[2] != "conv":   # stem ConvBNReLU already matches
            remapped[key] = value
            continue
        sub_idx, rest = parts[3], ".".join(parts[4:])
        has_expand = any(k.startswith(f"features.{block}.conv.6.") for k in state_dict)
        index_map = ({"0": "0.0", "1": "0.1", "3": "1.0", "4": "1.1", "6": "2", "7": "3"}
                     if has_expand else
                     {"0": "0.0", "1": "0.1", "3": "1", "4": "2"})
        if sub_idx not in index_map:              # ReLU6 -> no params
            continue
        new_key = f"features.{block}.conv.{index_map[sub_idx]}"
        remapped[new_key + ("." + rest if rest else "")] = value
    return remapped


def load_mobilenet_v2_035(weights_path=MNV2_035_WEIGHTS_PATH):
    """Return MobileNetV2 alpha=0.35 `.features`, initialised from the local checkpoint."""
    if not os.path.exists(weights_path):
        raise FileNotFoundError(
            f"MobileNetV2 alpha=0.35 weights not found at {weights_path}. "
            f"Upload mobilenetv2_0.35-*.pth to your Drive and update MNV2_035_WEIGHTS_PATH."
        )
    net = models.mobilenet_v2(weights=None, width_mult=0.35)
    state_dict = torch.load(weights_path, map_location="cpu")
    missing, _ = net.load_state_dict(_remap_mobilenet_v2_keys(state_dict), strict=False)
    # Everything except the unused final conv (features.18) must be populated.
    leftover = [k for k in missing
                if k.startswith("features.") and not k.startswith("features.18")]
    if leftover:
        raise RuntimeError(f"Checkpoint did not populate backbone layers: {leftover[:5]}")
    return net.features


# --- TARGET SMEARING (CenterNet-style soft Gaussian heatmaps) ---
# Smearing is applied ONLY to the railroad-crossing class (id 0): it is the one
# class whose signs rotate, so the exact centre cell is ambiguous on tilted
# examples and a tolerant Gaussian target helps the model fire. Its centroid is
# splatted as a 2D Gaussian (peak 1.0 at the centre cell, decaying outward).
# The OTHER classes are static (they do not rotate), so they keep a single hard
# cell (peak 1.0 at the centre cell only). The model has NO background channel:
# it predicts one sigmoid heatmap per class, trained with penalty-reduced focal
# loss. GAUSS_SIGMA is the smear width in GRID CELLS (~1 cell => bump spanning
# ~+/-3).
GAUSS_SIGMA = 1.0

# Only this class is smeared; see the note above.
RAILROAD_CROSSING_ID = 0

def draw_gaussian(heatmap, cx, cy, sigma):
    """Max-merge a 2D Gaussian (peak 1.0 at integer cell cx,cy) into heatmap[H,W]."""
    h, w = heatmap.shape
    radius = max(1, int(round(3 * sigma)))
    x0, x1 = max(0, cx - radius), min(w, cx + radius + 1)
    y0, y1 = max(0, cy - radius), min(h, cy + radius + 1)
    if x0 >= x1 or y0 >= y1:
        return
    ys = torch.arange(y0, y1, dtype=torch.float32).unsqueeze(1)
    xs = torch.arange(x0, x1, dtype=torch.float32).unsqueeze(0)
    g = torch.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2.0 * sigma * sigma))
    region = heatmap[y0:y1, x0:x1]
    heatmap[y0:y1, x0:x1] = torch.maximum(region, g)

def centernet_focal_loss(logits, target, eps=1e-6):
    """Penalty-reduced focal loss (CenterNet) on Gaussian heatmaps.

    logits: raw model output (B, C, H, W). target: Gaussian heatmap in [0,1].
    Cells where target == 1 are positives; others are negatives down-weighted by
    (1 - target)^4. Normalized by the number of positive (peak) cells. Static
    (un-smeared) classes are simply a hard 1.0 peak with 0 elsewhere, which this
    loss already handles as a standard positive cell surrounded by full-weight
    negatives.
    """
    pred = torch.sigmoid(logits).clamp(eps, 1.0 - eps)
    pos = target.eq(1.0).float()
    neg = 1.0 - pos
    neg_weights = torch.pow(1.0 - target, 4.0)
    pos_loss = torch.log(pred) * torch.pow(1.0 - pred, 2.0) * pos
    neg_loss = torch.log(1.0 - pred) * torch.pow(pred, 2.0) * neg_weights * neg
    num_pos = pos.sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()
    if num_pos == 0:
        return -neg_loss
    return -(pos_loss + neg_loss) / num_pos


class FOMO_PL_480(nn.Module):
    def __init__(self, num_classes, weights_path=MNV2_035_WEIGHTS_PATH):
        super(FOMO_PL_480, self).__init__()
        # MobileNetV2 alpha=0.35, initialised from the local ImageNet-pretrained checkpoint.
        backbone = load_mobilenet_v2_035(weights_path)

        # Cut at block_13 for 1/16 resolution (480 -> 30)
        # Cut at block_7 for 1/8 resolution (480 -> 60)
        self.features = nn.Sequential(*list(backbone.children())[:7])

        # Infer the cut's output channels (16 at alpha=0.35) instead of hardcoding,
        # so the head stays correct if the alpha / cut point ever changes.
        backbone_channels = self._infer_backbone_channels()

        # CenterNet-style head: one heatmap channel PER CLASS (no background).
        # Returns raw logits; sigmoid is applied in the loss and at decode.
        self.head = nn.Sequential(
            nn.Conv2d(in_channels=backbone_channels, out_channels=32, kernel_size=1, stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=num_classes, kernel_size=1, stride=1)
        )

    def _infer_backbone_channels(self):
        was_training = self.training
        self.eval()
        with torch.no_grad():
            channels = self.features(torch.zeros(1, 3, 64, 64)).shape[1]
        self.train(was_training)
        return channels

    def forward(self, x):
        return self.head(self.features(x))

class YOLOCentroidDataset(Dataset):
    def __init__(self, split_root, num_classes, img_size=480, grid_size=60, sigma=GAUSS_SIGMA):
        # Look for images and labels in the unzipped folder
        self.img_dir = os.path.join(split_root, "images")
        self.label_dir = os.path.join(split_root, "labels")
        self.img_paths = sorted([p for p in glob.glob(os.path.join(self.img_dir, "*"))
                                  if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.sigma = sigma
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label_path = os.path.join(self.label_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")

        # Target = one heatmap per class (C, grid, grid), no bg channel. The
        # railroad-crossing class (id 0) is smeared into a soft Gaussian bump;
        # the static classes are a single hard cell.
        gs = self.grid_size
        target = torch.zeros((self.num_classes, gs, gs), dtype=torch.float32)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 3: continue
                    cls_id, x_c, y_c = int(parts[0]), float(parts[1]), float(parts[2])
                    if not (0 <= cls_id < self.num_classes): continue
                    gx = min(int(x_c * gs), gs - 1)
                    gy = min(int(y_c * gs), gs - 1)
                    # Smear only the rotating railroad-crossing class; the other
                    # classes are static and get a single hard centre cell.
                    if cls_id == RAILROAD_CROSSING_ID:
                        draw_gaussian(target[cls_id], gx, gy, self.sigma)
                    else:
                        target[cls_id, gy, gx] = 1.0
        return self.transform(img), target

### Finetuning

In [ ]:
def run_training():
    # Setup Paths
    dataset_root = os.path.join(LOCAL_ROOT, "split_centroid_dataset")
    yaml_path = os.path.join(dataset_root, "data.yaml")
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)

    # Resolve paths relative to dataset_root
    train_path = os.path.join(dataset_root, "train")
    val_path = os.path.join(dataset_root, "val")

    # Set Device to CUDA (NVIDIA GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    num_classes = len(data_cfg['names'])

    train_ds = YOLOCentroidDataset(train_path, num_classes)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

    model = FOMO_PL_480(num_classes=num_classes).to(device)

    # Target smearing -> CenterNet penalty-reduced focal loss on sigmoid heatmaps.
    criterion = centernet_focal_loss
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 60
    for epoch in range(epochs):
        model.train()
        l_sum = 0
        for imgs, targs in train_loader:
            imgs, targs = imgs.to(device), targs.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            if preds.shape[-2:] != targs.shape[-2:]:
                raise RuntimeError(
                    f"Grid mismatch: model output {tuple(preds.shape[-2:])} vs target "
                    f"{tuple(targs.shape[-2:])}. Set grid_size to the model's output resolution."
                )
            loss = criterion(preds, targs)
            loss.backward()
            optimizer.step()
            l_sum += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {l_sum/len(train_loader):.4f}")

    # SAVE TO DRIVE
    save_path = "/content/drive/MyDrive/fomo_results/fomo_480.pt"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

run_training()